<a href="https://colab.research.google.com/github/poorvika12-hub/Educational-RAG-Chatbot/blob/main/Educational_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q google-generativeai

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
reader = PdfReader("OPERATING-SYSTEMS.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print(text[:1000])

In [ ]:
chunk_size = 500

chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])

print("Total Chunks:", len(chunks))

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
embeddings = model.encode(chunks)

print(embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors stored:", index.ntotal)

In [ ]:
question = input("Ask a question:  ")

In [ ]:
question_embedding = model.encode([question])

In [ ]:
k = 5

distances, indices = index.search(np.array(question_embedding), k)

print(indices)

In [ ]:
context = ""

for idx in indices[0]:
    context += chunks[idx] + "\n\n"

print(context)

In [ ]:
!pip install -q google-generativeai

In [ ]:
import google.generativeai as genai

In [ ]:
import os
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
for model in genai.list_models():
    print(model.name)

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model_gemini = genai.GenerativeModel("gemini-2.0-flash")

In [ ]:
prompt = f"""
You are an educational AI assistant.

Answer the question ONLY using the context provided below.

If the answer is not present in the context, say:
"I couldn't find that information in the uploaded PDF."

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
model_gemini = genai.GenerativeModel("gemini-flash-latest")

In [ ]:
prompt = f"""
You are an educational AI assistant.

Answer the question ONLY using the context below.

If the answer is not present in the context, say:
"I couldn't find the answer in the uploaded document."

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
question_embedding = embedding_model.encode([question])

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

gemini_model = genai.GenerativeModel("gemini-flash-latest")

In [ ]:
print(type(model))

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
embeddings = embedding_model.encode(chunks)

In [ ]:
question_embedding = embedding_model.encode([question])

In [ ]:
k = 5

distances, indices = index.search(np.array(question_embedding), k)

print(indices)

In [ ]:
context = ""

for idx in indices[0]:
    context += chunks[idx] + "\n\n"

print(context)

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

gemini_model = genai.GenerativeModel("gemini-flash-latest")

In [ ]:
prompt = f"""
Answer only using the following context.

Context:
{context}

Question:
{question}

Answer:
"""

response = gemini_model.generate_content(prompt)

print(response.text)

In [ ]:
print(type(model))

In [ ]:
while True:

    question = input("\nAsk your question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("Thank you for using the RAG Chatbot!")
        break

    # Create embedding for the question
    question_embedding = embedding_model.encode([question])

    # Retrieve top 5 chunks
    distances, indices = index.search(np.array(question_embedding), 5)

    # Build context
    context = ""

    for idx in indices[0]:
        context += chunks[idx] + "\n\n"

    # Create prompt
    prompt = f"""
You are an educational AI assistant.

Answer ONLY using the context below.

If the answer is not available in the context, say:
'I couldn't find that information in the uploaded PDF.'

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer
    response = gemini_model.generate_content(prompt)

    print("\nAnswer:")
    print(response.text)
    print("=" * 80)